In [1]:
import jax
import jax.numpy as jnp
import qcsys as qs
import scqubits
import numpy as np
import qutip

jax.config.update('jax_platform_name', 'cpu')
# jax.config.update('jax_enable_x64', True)


# fluxonium: operator has difference due to difference in eigen vector calculated from jax/scipy

In [3]:

qsf = qs.Fluxonium.create(
    25,
    {"Ej": 2.7, "Ec": 0.6, "El": 0.13, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
)

scf = scqubits.Fluxonium(EJ=2.7,
                        EC=0.6,
                        EL=0.13,
                        flux=0,cutoff=100,
                        truncated_dim=25)


np.array_equal(np.array(scf.n_operator())[:25,:25] , np.array(qsf.get_op_in_H_eigenbasis(qsf.linear_ops['n']).data)), \
    np.allclose(np.array(qutip.Qobj(scf.hamiltonian()).tidyup()), np.array(qsf.get_H_full().data,dtype = 'complex128'),atol=1e-08)


(False, True)

# transmon

In [4]:
qst_sc = qs.SingleChargeTransmon.create(
    N = 6,
    params = {"Ej": 40, "Ec": 0.5,"ng":0.0},
    N_max_charge=30
)

sct = scqubits.Transmon(
    EJ=40,
    EC=0.5,
    ng=0.0,
    ncut=30,
    truncated_dim = 6
    )

np.array_equal(np.array(sct.n_operator()),np.array(qst_sc.linear_ops['n'].data)), \
    np.array_equal(np.array(sct.hamiltonian()),np.array(qst_sc.get_H_full().data))

(True, True)

# hilbertspace: bare hamiltonian are the same 

In [5]:
import jaxquantum as jqt
devices = [qsf, qst_sc]
f_indx = 0
t_indx = 1
Ns = [device.N for device in devices]
tn = qs.promote(qst_sc.ops['n'], t_indx, Ns)
fn = qs.promote(qsf.ops["n"], f_indx, Ns)
g_tf = 0.2
system = qs.System.create(devices, couplings=[
    g_tf *  fn @ tn
])
system.params["g_tf"] = g_tf
Es, kets = system.calculate_eig_linear()


hilbertspace = scqubits.HilbertSpace([scf, sct])
hilbertspace.add_interaction(g_strength=g_tf,op1=scf.n_operator, op2=sct.n_operator,add_hc=False)
evals, evecs = hilbertspace.hamiltonian().eigenstates()


# Tmon part bare H
evals = sct.eigenvals(evals_count=sct.truncated_dim)
scq_tmon_bare = np.array(hilbertspace.diag_hamiltonian(sct, evals).full())

I_ops = [jqt.identity(N) for N in system.Ns]
qcs_tmon_bare = np.array(system.promote(qst_sc.get_H(), 1).data)

# Fluxonium part bare H
evals = scf.eigenvals(evals_count=scf.truncated_dim)
scq_f_bare = np.array(hilbertspace.diag_hamiltonian(scf, evals).full())

I_ops = [jqt.identity(N) for N in system.Ns]
qcs_f_bare = np.array(system.promote(qsf.get_H(), 0).data)


np.allclose(scq_f_bare,qcs_f_bare), \
    np.allclose(scq_tmon_bare,qcs_tmon_bare), \
        np.allclose(np.array(system.get_H_bare().data), np.array(hilbertspace.bare_hamiltonian()))

(True, True, True)

# hilbertspace: interaction hamiltonians or full hamiltonians have ONLY sign differences

In [6]:
np.allclose(np.array(qutip.Qobj(np.array(system.get_H_couplings().data)).tidyup()),
             np.array(qutip.Qobj(np.array(hilbertspace.interaction_hamiltonian())).tidyup())), \
    np.sum(np.abs(np.array(qutip.Qobj(np.array(system.get_H_couplings().data)).tidyup())) 
       - np.abs(np.array(qutip.Qobj(np.array(hilbertspace.interaction_hamiltonian())).tidyup())) > 1e-6)

(False, 0)

In [7]:
np.allclose(np.array(qutip.Qobj(np.array(system.get_H().data)).tidyup()),
             np.array(qutip.Qobj(np.array(hilbertspace.hamiltonian())).tidyup())), \
    np.sum(np.abs(np.array(qutip.Qobj(np.array(system.get_H().data)).tidyup())) 
       - np.abs(np.array(qutip.Qobj(np.array(hilbertspace.hamiltonian())).tidyup())) > 1e-6)

(False, 0)

# Hillbertspace eigenvals are the same, and eigenvectors only have sign difference

In [8]:
evals, evecs = system.calculate_eig_linear()
sc_evals, sc_evecs = hilbertspace.eigensys(150)
hilbertspace.generate_lookup()
for f in range(25):
    for t in range(6):
        assert np.allclose(evals[f,t].item(),sc_evals[hilbertspace.dressed_index((f,t))])

In [9]:
for f in range(25):
    for t in range(6):
        assert np.allclose(np.abs(np.array(evecs[f,t]).reshape(-1,1)),np.abs(np.array(sc_evecs[hilbertspace.dressed_index((f,t))]).reshape(-1,1)),atol = 1e-8), f"{f},{t}"

# Dressed ops have only sign differences

In [49]:
# Step-1 id wrapped operators are close

import scqubits.utils.misc as utils
import scqubits.utils.spectrum_utils as spec_utils

truncated_dim = hilbertspace.dimension
op = sct.n_operator
op_in_bare_eigenbasis = False
subsys_index = hilbertspace.get_subsys_index(op.__self__)
bare_evecs = hilbertspace._data["bare_evecs"][subsys_index][0]

id_wrapped_op_sc = spec_utils.identity_wrap(
    op,
    hilbertspace.subsystem_list[subsys_index],
    hilbertspace.subsystem_list,
    op_in_eigenbasis=op_in_bare_eigenbasis,
    evecs=bare_evecs,
)
id_wrapped_op_qs = np.array(tn.data)

np.allclose(np.abs(id_wrapped_op_sc.full()), np.abs(id_wrapped_op_qs))

True

In [54]:
# Step-2 the eigenvalues useed for transform are close


def calculate_eig(H: jqt.Qarray):
    vals, kets = jnp.linalg.eigh(H.data)
    return (vals,kets) # Here kets is equivalent to the S in qutip.Qobj.transform


import scipy.sparse
S_qs = np.array(calculate_eig(system.get_H())[1].T)

dressed_evecs = hilbertspace._data["evecs"][0]
S_sc = (scipy.sparse.hstack([psi.data for psi in dressed_evecs], format='csr', dtype=complex).conj().T).toarray()

np.sum(np.abs(S_sc) - np.abs(S_qs) > 1e-10)

0

In [59]:
def transform_op_into_dressed_basis_jax(op_matrix: jqt.Qarray, 
                                        S: jax.Array) -> jax.Array:
    """
    Transform an operator into the dressed basis using JAX.

    Parameters:
    - op_matrix: A 2D JAX array representing the operator's matrix.
    - S: A 2D JAX array representing the dressed eigenvectors similar to the S in qutip.Qobj.transform

    Returns:
    - A 2D JAX array representing the transformed operator.
    """
    data = jnp.dot(S, jnp.dot(op_matrix.data, S.T.conj()))
    return data


qs_n = transform_op_into_dressed_basis_jax(tn, calculate_eig(system.get_H())[1].T)

sc_n = hilbertspace.op_in_dressed_eigenbasis(sct.n_operator)
np.allclose(np.abs((np.array(qs_n))) , np.abs(sc_n.full()))

True